# Frequent Itemset Mining: what sells together?
## The campus café, 500 orders
**Salem Othman · Student notebook**

Classification needed an answer column. Today there is none. You get 500 café orders and one question: **which items are bought together, and which of those patterns are real?**

You will build the basket table, measure how often itemsets occur, use the Apriori idea to avoid checking every possible combination, and turn itemsets into rules a manager could act on. The last part is an audit: an AI recommendation that sounds convincing and is wrong.

### Today's five steps
| Part | Step | What you do | Exercise |
|---|---|---|---|
| 1 | Baskets | Turn a list of order lines into one row per order | E1 |
| 2 | Support | Measure how often an itemset appears | E2 |
| 3 | Apriori | Find frequent pairs without checking every pair | E3 |
| 4 | Rules | Measure confidence and lift for "if A then B" | E4 |
| 5 | Audit | Check an AI recommendation against your numbers | E5 |

**For every exercise:** predict → write the code → check → explain.

**How to work**
- Run cells from top to bottom with **Shift + Enter**.
- In an exercise cell, **write the code yourself** under each numbered step. Keep the variable names exactly as written; the Check cell looks for them.
- `PASS` means that step is done. `FIX` tells you what to look at.
- If a Check cell says an exercise **is not finished yet**, that is not an error: write the code in the exercise cell above, run it, then run the Check cell again.

In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def check(label, passed):
    """Print PASS or FIX for one requirement."""
    print(("PASS  " if passed else "FIX   ") + label)

def ready(value, step):
    """True when the exercise above is done. Otherwise print a reminder and return False.

    It never raises, so a blank exercise cell can never stop the class with a red
    traceback: the Check cell just tells the student what to do next.
    """
    if value is None:
        print(f"{step} is not finished yet.")
        print("Write your code in the exercise cell above, run that cell, then run this one again.")
        return False
    return True

print("Setup complete.")

Setup complete.


# Part 0 · The orders
Every row below is **one item in one order**. Order O004 bought a bagel, a coffee and a sandwich, so it has three rows. This is how a till system really stores sales.

There is no answer column. Nobody has labelled these orders; we are looking for patterns, not predicting a target. That makes today **unsupervised**: no train/test split, no accuracy.

In [2]:
rng = np.random.default_rng(11)
n_orders = 500
rows = []
for number in range(1, n_orders + 1):
    items = set()
    if rng.random() < 0.80: items.add("coffee")
    if rng.random() < 0.12: items.add("tea")
    if rng.random() < 0.20:
        items.add("bagel")
        if rng.random() < 0.70: items.add("cream_cheese")
    elif rng.random() < 0.02: items.add("cream_cheese")
    if rng.random() < 0.18: items.add("muffin")
    if rng.random() < 0.22:
        items.add("sandwich")
        if rng.random() < 0.55: items.add("chips")
    elif rng.random() < 0.06: items.add("chips")
    if rng.random() < 0.10:
        items.add("salad")
        if rng.random() < 0.60: items.add("water")
    elif rng.random() < 0.07: items.add("water")
    if rng.random() < 0.09: items.add("cookie")
    if not items: items.add("coffee")
    for item in sorted(items):
        rows.append((f"O{number:03d}", item))

orders = pd.DataFrame(rows, columns=["order_id", "item"])
print("Order lines:", len(orders), "| Orders:", orders["order_id"].nunique())
display(orders.head(8))

Order lines: 1089 | Orders: 500


,order_id,item
0,O001,coffee
1,O001,muffin
2,O002,coffee
3,O002,muffin
4,O003,coffee
5,O004,bagel
6,O004,coffee
7,O004,sandwich


# Part 1 · From order lines to baskets
To count what is bought together we need **one row per order** and **one column per item**, with `True` where the order contains that item. This is the **basket table** (also called a transaction table or one-hot table).

| order_id | bagel | coffee | sandwich | ... |
|---|---|---|---|---|
| O004 | True | True | True | ... |

`pd.crosstab(orders["order_id"], orders["item"])` counts each (order, item) pair; `> 0` turns the counts into True/False.

**Support of an item** = the share of orders that contain it. If 438 of 500 orders contain coffee, support(coffee) = 0.876.

### E1 · Build the basket table
**Predict:** which item do you expect in the most orders? Roughly what share?

In [3]:
basket = None
items_per_order = None
item_support = None

# Write your code below. Your code must create the variables set to None above.

# 1. basket: one row per order, one column per item, True where the order has the item.
#    Hint: pd.crosstab(orders["order_id"], orders["item"]) > 0

# 2. items_per_order: how many items each order contains.
#    Hint: basket.sum(axis=1)   (True counts as 1)
#    Print its average.

# 3. item_support: the share of orders that contain each item, highest first.
#    Hint: basket.mean() gives the share of True in each column; then .sort_values(ascending=False)

# 4. Print the shape of basket and display item_support.

**Check E1**

In [4]:
if ready(basket, "E1") and ready(item_support, "E1"):
    check("basket has one row per order", len(basket) == orders["order_id"].nunique())
    check("basket has one column per item", basket.shape[1] == orders["item"].nunique())
    check("basket holds True/False values", basket.dtypes.eq(bool).all())
    check("items_per_order adds up to the number of order lines", items_per_order.sum() == len(orders))
    check("item_support is sorted from the most common item down", item_support.is_monotonic_decreasing)
    check("every support is a share between 0 and 1", item_support.between(0, 1).all())

E1 is not finished yet.
Write your code in the exercise cell above, run that cell, then run this one again.


**Write:** coffee is in most orders. Does that make coffee interesting for finding patterns? Why or why not?

*Your answer:*

# Part 2 · Support of an itemset
An **itemset** is any group of items, like {bagel, cream_cheese}. Its support is the share of orders that contain **all** of them.

In the basket table, an order contains both bagel and cream cheese when both columns are True:

`basket[["bagel", "cream_cheese"]].all(axis=1)` is True only for those orders, and `.mean()` turns that into a share.

A rule of thumb you will use all week: **an itemset is frequent when its support is at least the minimum support** we choose (today 0.10, so at least 50 of 500 orders).

### E2 · A support function
**Predict:** support(bagel) is 0.182 and support(cream_cheese) is 0.144. Can support({bagel, cream_cheese}) be larger than 0.144?

*Your prediction:*

In [5]:
support = None
pair_support = None

# Write your code below. Your code must create the variables set to None above.

# 1. support: a function that takes a list of item names and returns the share of
#    orders containing ALL of them.
#    Hint:
#    def support(items):
#        return basket[items].all(axis=1).mean()

# 2. pair_support: support(["bagel", "cream_cheese"]).
#    Print it, and print how many orders that is (multiply by len(basket)).

# 3. Print support(["coffee"]) and support(["coffee", "sandwich"]) to see the function work
#    on one item and on a pair.

**Check E2**

In [6]:
if ready(support, "E2") and ready(pair_support, "E2"):
    check("support of one item matches item_support",
          np.isclose(support(["coffee"]), basket["coffee"].mean()))
    check("pair_support is the share of orders with both items",
          np.isclose(pair_support, (basket["bagel"] & basket["cream_cheese"]).mean()))
    check("a pair is never more common than its rarer item",
          pair_support <= min(support(["bagel"]), support(["cream_cheese"])))
    check("support works on three items too", 0 <= support(["coffee", "bagel", "cream_cheese"]) <= pair_support)

E2 is not finished yet.
Write your code in the exercise cell above, run that cell, then run this one again.


**Write:** support({bagel, cream_cheese}) is a little below support(cream_cheese). What does that tell you about cream cheese buyers?

*Your answer:*

# Part 3 · Apriori: skip what cannot be frequent
With 10 items there are 45 possible pairs, 120 triples, and 1,023 non-empty itemsets in total. A real shop has thousands of items, so checking every itemset is hopeless.

**The Apriori rule:** if an itemset is not frequent, no bigger itemset containing it can be frequent either. Adding an item can only keep the support the same or make it smaller (you saw that in E2).

So we work upward and throw away dead ends early:
1. Find the **frequent items** (support at least 0.10).
2. Build **candidate pairs only from frequent items**.
3. Keep the candidate pairs that are frequent.
4. Build candidate triples only from frequent pairs, and so on.

A pair that includes salad (support 0.076) cannot reach 0.10, so we never even count it.

### E3 · Frequent items and frequent pairs
**Predict:** with a minimum support of 0.10, how many of the 10 items are frequent? How many of the 45 possible pairs will we still have to check?

*Your prediction:*

In [7]:
frequent_items = None
candidate_pairs = None
frequent_pairs = None

MIN_SUPPORT = 0.10

# Write your code below. Your code must create the variables set to None above.

# 1. frequent_items: a sorted list of the items whose support is at least MIN_SUPPORT.
#    Hint: item_support[item_support >= MIN_SUPPORT].index, then sorted(...)

# 2. candidate_pairs: every pair of frequent items, as a list of tuples.
#    Hint: list(combinations(frequent_items, 2))

# 3. frequent_pairs: a DataFrame with one row per candidate pair whose support is at least
#    MIN_SUPPORT, and the columns item_a, item_b, support, sorted by support (highest first).
#    Hint: loop over candidate_pairs, call support([a, b]), keep the ones that pass.

# 4. Print how many pairs exist in total (all 10 items), how many candidates you checked,
#    and how many are frequent. Display frequent_pairs.

**Check E3**

In [8]:
if ready(frequent_items, "E3") and ready(frequent_pairs, "E3"):
    check("frequent_items holds exactly the items at or above the minimum support",
          set(frequent_items) == set(basket.columns[basket.mean() >= MIN_SUPPORT]))
    check("candidates were built from frequent items only",
          all(a in frequent_items and b in frequent_items for a, b in candidate_pairs))
    check("Apriori checked fewer pairs than the brute-force count",
          len(candidate_pairs) < len(list(combinations(basket.columns, 2))))
    check("frequent_pairs has item_a, item_b and support",
          set(frequent_pairs.columns) >= {"item_a", "item_b", "support"})
    check("every pair kept reaches the minimum support", (frequent_pairs["support"] >= MIN_SUPPORT).all())
    check("no frequent pair was missed",
          len(frequent_pairs) == sum(support([a, b]) >= MIN_SUPPORT for a, b in candidate_pairs))

E3 is not finished yet.
Write your code in the exercise cell above, run that cell, then run this one again.


### Worked example · One level up: triples
The same rule again: a triple can only be frequent if **all three of its pairs** are frequent. From 7 frequent pairs, only two triples survive that test.

In [9]:
if ready(frequent_pairs, "E3"):
    pair_set = {frozenset((a, b)) for a, b in zip(frequent_pairs["item_a"], frequent_pairs["item_b"])}
    items_in_pairs = sorted(set(frequent_pairs["item_a"]) | set(frequent_pairs["item_b"]))

    candidate_triples = [t for t in combinations(items_in_pairs, 3)
                         if all(frozenset(p) in pair_set for p in combinations(t, 2))]
    print("Possible triples of all 10 items:", len(list(combinations(basket.columns, 3))))
    print("Triples worth checking:", len(candidate_triples))
    for t in candidate_triples:
        print("  ", t, "support", round(support(list(t)), 3))

E3 is not finished yet.
Write your code in the exercise cell above, run that cell, then run this one again.


120 possible triples, 2 worth checking. Both pass (0.102 and 0.118), and both are "breakfast with coffee" and "lunch with coffee". That is Apriori's whole value: the work shrinks as the itemsets grow.

**Write:** Apriori never counted the pair {salad, water}. Is it possible that salad and water really do go together? What did the minimum support decide for us?

*Your answer:*

# Part 4 · Rules: confidence and lift
A frequent pair {bagel, cream_cheese} becomes a **rule** when we read it in one direction: **if** an order has cream cheese, **then** it has a bagel. Written `cream_cheese → bagel`.

Two numbers judge a rule:

| Measure | Formula | Plain question |
|---|---|---|
| **Confidence** | support(A and B) / support(A) | Of the orders with A, what share also have B? |
| **Lift** | confidence / support(B) | Is B more likely *because* A is in the basket? |

**Reading lift:**
- **lift > 1:** A makes B more likely than usual. A real association.
- **lift ≈ 1:** A tells you nothing about B. They are independent.
- **lift < 1:** A makes B *less* likely than usual.

Confidence alone can mislead when B is common. If 88% of all orders have coffee, then "if anything, then coffee" will have high confidence no matter what.

### E4 · Confidence and lift for every frequent pair
**Predict:** five of the seven frequent pairs contain coffee. Will **any** rule involving coffee have a lift above 1?

*Your prediction:*

In [10]:
rules = None

# Write your code below. Your code must create the variable set to None above.

# 1. For every row of frequent_pairs, make TWO rules: item_a -> item_b and item_b -> item_a.
#    For each rule compute:
#       support    = support([if_item, then_item])
#       confidence = support / support([if_item])
#       lift       = confidence / support([then_item])
#    Hint: loop over zip(frequent_pairs["item_a"], frequent_pairs["item_b"]),
#    then over [(a, b), (b, a)].

# 2. rules: a DataFrame with the columns if_item, then_item, support, confidence, lift,
#    sorted by lift (highest first). Round the numbers to 3 decimals.

# 3. Display rules.

**Check E4**

In [11]:
if ready(rules, "E4"):
    check("rules has two rules for every frequent pair", len(rules) == 2 * len(frequent_pairs))
    check("rules reports if_item, then_item, support, confidence and lift",
          set(rules.columns) >= {"if_item", "then_item", "support", "confidence", "lift"})
    check("rules is sorted by lift, highest first", rules["lift"].is_monotonic_decreasing)
    row = rules[(rules["if_item"] == "cream_cheese") & (rules["then_item"] == "bagel")]
    check("confidence = support of the pair / support of the 'if' item",
          len(row) == 1 and np.isclose(row["confidence"].iloc[0],
                                       support(["cream_cheese", "bagel"]) / support(["cream_cheese"]), atol=0.001))
    check("a rule and its reverse have the same lift",
          rules.groupby(rules[["if_item", "then_item"]].apply(lambda r: frozenset(r), axis=1))["lift"].nunique().eq(1).all())

E4 is not finished yet.
Write your code in the exercise cell above, run that cell, then run this one again.


**Write:** the café manager can run one promotion. Which rule from your table would you pick, and which high-confidence rule would you warn them against? Use the numbers.

*Your answer:*

# Part 5 · Audit the AI recommendation
The café manager asks an AI assistant to analyse the orders. It replies:

> "81% of customers who buy a sandwich also buy a coffee. That is a strong association, so you should place a coffee promotion next to the sandwich display. I used a minimum support of 10%, and no other strong patterns exist below that level, so you can ignore rare items like salads."

It sounds careful. Check it.

### E5 · Check both claims

In [12]:
sandwich_coffee = None
salad_water = None

# Write your code below. Your code must create the variables set to None above.

# 1. sandwich_coffee: a dictionary with the support, confidence and lift of sandwich -> coffee.
#    Use the same formulas as E4.

# 2. salad_water: the same three numbers for salad -> water.
#    Note: this pair is below MIN_SUPPORT, so it is NOT in your rules table. Compute it directly.

# 3. Print both dictionaries, rounded to 3 decimals.
#    Hint: float(round(x, 3)) prints as a plain number.

**Check E5**

In [13]:
if ready(sandwich_coffee, "E5") and ready(salad_water, "E5"):
    check("sandwich -> coffee has the confidence the AI quoted (about 81%)",
          abs(sandwich_coffee["confidence"] - 0.81) < 0.01)
    check("its lift is below 1", sandwich_coffee["lift"] < 1)
    check("salad -> water is below the 10% minimum support", salad_water["support"] < 0.10)
    check("salad -> water has a higher lift than any rule above the threshold",
          ready(rules, "E4") and salad_water["lift"] > rules["lift"].max())

E5 is not finished yet.
Write your code in the exercise cell above, run that cell, then run this one again.


**Write:** answer the AI in three or four sentences, one point for each of its two claims, using your numbers.

*Your answer:*

### Challenge tasks (optional)
1. Lower `MIN_SUPPORT` to 0.04 and rerun E3 and E4. How many frequent pairs now? Which new rules have lift above 3?
2. Write the rule `{bagel, coffee} → cream_cheese` and compute its confidence and lift. Does adding coffee to the "if" side change anything?
3. If you want a library later: `mlxtend.frequent_patterns.apriori(basket, min_support=0.10, use_colnames=True)` gives the same frequent itemsets. Install it with `pip install mlxtend` and compare with your table.

# Exit ticket
Answer without running more code.
1. support(A) = 0.30 and support(A and B) = 0.12. What is the confidence of A → B?
2. A rule has confidence 0.85 and lift 0.95. Is it a useful pattern? Why?
3. Why can Apriori skip a pair that contains an infrequent item without counting it?

*Your answers:*
1.
2.
3.

# Submit
- Restart the kernel and run all cells. Every check should print `PASS`.
- Every *Your answer* and *Your prediction* cell is filled in.
- Submit this notebook with its outputs.

**Reminder:** Exam 1 is on Friday and covers weeks 1–4 (preprocessing and classification). Today's topic is on Exam 2.